# 06. Regressions

Table 2: ordinary least squares of four climate outcomes on the country-level indices, with HC3 standard errors. Panel A enters ADI and OSI jointly; Panels B and C estimate one model per thematic group; Panel D reports the controls from the Panel A model. All continuous variables are standardized and outcomes are logged.


In [1]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), 'src'))  # the modules the notebooks import live in src/


In [2]:
import os
import warnings

import numpy as np
import pandas as pd

import config as cfg
import aggregate as ag
import regress as rg

warnings.filterwarnings('ignore', category=FutureWarning)
O = cfg.OUTPUT_DIR
T = os.path.join(O, 'tables')
os.makedirs(T, exist_ok=True)
pd.set_option('display.width', 220)
print("output:", O)


output: …/output


## Covariates


In [3]:
cov = rg.load_covariates(2021)
print(f"{len(cov)} countries")
print(cov[list(rg.OUTCOMES.values()) + rg.CONTROLS].notna().sum().to_string())
print()
print(cov[rg.CONTROLS].describe().round(3).to_string())

218 countries
ghg_per_capita                  196
co2_per_capita                  213
co2_including_luc_per_capita    194
energy_per_capita               204
ln_gdp_pc                       164
fossil_share                    201
gov_effectiveness               200
vulnerability                   187

       ln_gdp_pc  fossil_share  gov_effectiveness  vulnerability
count    164.000       201.000            200.000        187.000
mean       9.236        59.224              0.028          0.433
std        1.216        33.164              0.995          0.093
min        6.538         0.000             -2.243          0.253
25%        8.376        27.830             -0.690          0.365
50%        9.414        66.200             -0.051          0.421
75%       10.191        88.290              0.732          0.508
max       11.776       100.000              2.218          0.636


## Country-level indices


In [4]:
# Country-level indices. Rebuilt from the post-level files when they are present;
# otherwise the released indices are read (data/indices_2022_by_country.csv, copied to
# output/). Either way the frame is the one Table 2 is estimated on.
POSTS = os.path.join(O, 'climate_posts_2022.parquet')
if os.path.exists(POSTS):
    posts = pd.read_parquet(os.path.join(O, 'climate_posts_2022.parquet'),
                            columns=['tweet_id', 'country'])
    recall = pd.read_parquet(os.path.join(O, 'topic_recall_2022.parquet'))
    stance = pd.read_parquet(os.path.join(O, 'topic_stance_2022.parquet'))
    # the post's own stance, which OSI is the plain mean of
    sscore = pd.read_parquet(os.path.join(O, 'stance_2022.parquet'),
                             columns=['tweet_id', 'stance_score'])
    assert (sscore.tweet_id.to_numpy() == stance.tweet_id.to_numpy()).all()
    sscore = sscore.stance_score
    ga_chunks = pd.read_csv(os.path.join(O, 'GA_topics.csv'), low_memory=False)
    
    assert (posts.tweet_id.to_numpy() == recall.tweet_id.to_numpy()).all()
    assert (posts.tweet_id.to_numpy() == stance.tweet_id.to_numpy()).all()
    
    posts = ag.add_party(posts)
    ga_party = ag.ndc_by_party(ga_chunks)
    ga_country = rg.ga_by_country(ga_party)
    
    by_country, meta_c = ag.country_indices(posts.shp_country, recall, stance, ga_country,
                                            stance_score=sscore, label='per country')
    by_party, meta_p = ag.country_indices(posts.party, recall, stance, ga_party,
                                          stance_score=sscore, label='per Party')
    by_country.to_csv(os.path.join(O, 'indices_2022_by_country.csv'))
    print(f"per country {len(by_country):>4} units, {meta_c['posts']:,} posts")
    print(f"per Party   {len(by_party):>4} units, {meta_p['posts']:,} posts")
    
    eu = [c for c in cfg.EU_MEMBER_COUNTRIES if c in by_country.index]
    print(f"\nEU members above the threshold: {len(eu)}")
    print(f"  their own ADI spans {by_country.loc[eu, 'ADI'].min():.4f} to "
          f"{by_country.loc[eu, 'ADI'].max():.4f}, "
          f"against the EU's pooled {by_party.loc['EU', 'ADI']:.4f}")
else:
    by_country = pd.read_csv(os.path.join(O, 'indices_2022_by_country.csv'), index_col=0)
    print(f"read released indices: {len(by_country)} countries")


read released indices: 100 countries


## Table 2


In [5]:
G_TVR = [f'G{i}_TVR' for i in range(1, 7)]
G_OSI = [f'G{i}_OSI' for i in range(1, 7)]
COV_YEAR = 2021   # covariate year used for Table 2


def one_at_a_time(panel, predictors):
    """One regression per thematic group (Panels B and C), each with all controls."""
    c, s, p = {}, {}, {}
    for v in predictors:
        cc, ss, pp, _ = rg.fit_table1(panel, predictors=[v])
        c[v], s[v], p[v] = cc.loc[v], ss.loc[v], pp.loc[v]
    return pd.DataFrame(c).T, pd.DataFrame(s).T, pd.DataFrame(p).T


In [6]:
panel_new = rg.build_panel(by_country, year=COV_YEAR, level='party')
coef_a, se_a, pv_a, meta_a = rg.fit_table1(panel_new)
cb, sb, pb = one_at_a_time(panel_new, G_TVR)
cc, sc_, pc = one_at_a_time(panel_new, G_OSI)

print(f"{len(panel_new)} countries in, N = {int(meta_a.N.iloc[0])} after covariates\n")
print("Panel A. Aggregate")
print(rg.format_table(coef_a.loc[['ADI', 'OSI']], se_a.loc[['ADI', 'OSI']],
                      pv_a.loc[['ADI', 'OSI']], meta_a).to_string())
print()
print("Panel B. Topic-level (TVR), one regression per group")
print(rg.format_table(cb, sb, pb, meta_a).drop(index=['N', 'R2']).to_string())
print()
print("Panel C. Topic-level (OSI), one regression per group")
print(rg.format_table(cc, sc_, pc, meta_a).drop(index=['N', 'R2']).to_string())
print()
print("Panel D. Controls")
print(rg.format_table(coef_a.loc[rg.CONTROLS], se_a.loc[rg.CONTROLS],
                      pv_a.loc[rg.CONTROLS], meta_a).to_string())


100 countries in, N = 96 after covariates

Panel A. Aggregate
          GHG per capita CO2 per capita CO2 incl. LUC Energy per capita
Variables                                                              
ADI              0.122**       0.137***      0.154***          0.183***
                 (0.049)        (0.044)       (0.055)           (0.064)
OSI               -0.072         -0.111        -0.105            -0.094
                 (0.085)        (0.071)       (0.091)           (0.081)
N                     96             96            96                96
R2                  0.64            0.9          0.63              0.92

Panel B. Topic-level (TVR), one regression per group
          GHG per capita CO2 per capita CO2 incl. LUC Energy per capita
Variables                                                              
G1_TVR         -0.143***       -0.115**     -0.163***         -0.152***
                 (0.049)        (0.048)       (0.060)           (0.048)
G2_TVR          0.14

## Save


In [7]:
def header(name):
    return pd.DataFrame([{'Variables': name}]).set_index('Variables')

full = pd.concat([
    header('Panel A. Aggregate'),
    rg.format_table(coef_a.loc[['ADI', 'OSI']], se_a.loc[['ADI', 'OSI']],
                    pv_a.loc[['ADI', 'OSI']], meta_a).drop(index=['N', 'R2']),
    header('Panel B. Topic-level (TVR)'),
    rg.format_table(cb, sb, pb, meta_a).drop(index=['N', 'R2']),
    header('Panel C. Topic-level (OSI)'),
    rg.format_table(cc, sc_, pc, meta_a).drop(index=['N', 'R2']),
    header('Panel D. Controls'),
    rg.format_table(coef_a.loc[rg.CONTROLS], se_a.loc[rg.CONTROLS],
                    pv_a.loc[rg.CONTROLS], meta_a),
])
full.to_csv(os.path.join(T, 'Table2.csv'))
panel_new.to_csv(os.path.join(O, 'regression_panel_2022.csv'))
print("wrote")
print("  tables/Table2.csv")
print("  regression_panel_2022.csv")
full


wrote
  tables/Table2.csv
  regression_panel_2022.csv


,GHG per capita,CO2 per capita,CO2 incl. LUC,Energy per capita
Variables,,,,
Panel A. Aggregate,NaN,NaN,NaN,NaN
ADI,0.122**,0.137***,0.154***,0.183***
,(0.049),(0.044),(0.055),(0.064)
OSI,-0.072,-0.111,-0.105,-0.094
,(0.085),(0.071),(0.091),(0.081)
Panel B. Topic-level (TVR),NaN,NaN,NaN,NaN
G1_TVR,-0.143***,-0.115**,-0.163***,-0.152***
,(0.049),(0.048),(0.060),(0.048)
G2_TVR,0.148***,0.183***,0.174***,0.191***
